# 02 - LangChain Básico

En este notebook aprenderemos:
- Prompt Templates: plantillas reutilizables
- Chains: encadenar operaciones
- Output Parsers: estructurar la salida del modelo
- LCEL (LangChain Expression Language): la forma moderna de componer

In [1]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="gemma3:12b", temperature=0)

## 2.1 Prompt Templates

En lugar de construir strings manualmente, usamos plantillas con variables.

In [2]:
from langchain_core.prompts import ChatPromptTemplate

# Plantilla con variables
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un experto en {tema}. Responde de forma clara y concisa."),
    ("human", "{pregunta}"),
])

# Renderizar la plantilla
mensajes = prompt.invoke({"tema": "historia colombiana", "pregunta": "¿Qué pasó en 1810?"})
print("Mensajes generados:")
for m in mensajes.messages:
    print(f"  [{m.type}]: {m.content}")

Mensajes generados:
  [system]: Eres un experto en historia colombiana. Responde de forma clara y concisa.
  [human]: ¿Qué pasó en 1810?


## 2.2 Chains con LCEL (pipe operator)

LCEL permite componer componentes con el operador `|` (pipe), como en bash.

In [3]:
from langchain_core.output_parsers import StrOutputParser

# Chain: prompt → modelo → parser
chain = prompt | llm | StrOutputParser()

# Ejecutar la chain
resultado = chain.invoke({"tema": "geografía", "pregunta": "¿Cuál es el río más largo de Colombia?"})
print(resultado)

El río más largo de Colombia es el **río Amazonas**. Aunque gran parte de su recorrido está en Brasil, su nacimiento se da en Colombia, específicamente en el nevado de los Andes, donde se unen los ríos Maripá y Caquetá.



## 2.3 Output Parsers: salida estructurada

Podemos pedirle al modelo que devuelva datos en formato JSON estructurado usando Pydantic.

In [ ]:
from pydantic import BaseModel, Field


class Ciudad(BaseModel):
    """Información sobre una ciudad."""
    nombre: str = Field(description="Nombre de la ciudad")
    pais: str = Field(description="País donde se encuentra")
    poblacion_aprox: str = Field(description="Población aproximada")
    dato_curioso: str = Field(description="Un dato curioso sobre la ciudad")


# with_structured_output fuerza al modelo a devolver el esquema Pydantic
llm_estructurado = llm.with_structured_output(Ciudad)

resultado = llm_estructurado.invoke("Cuéntame sobre Medellín")
print(f"Ciudad: {resultado.nombre}")
print(f"País: {resultado.pais}")
print(f"Población: {resultado.poblacion_aprox}")
print(f"Dato curioso: {resultado.dato_curioso}")

## 2.4 Chains secuenciales

Podemos encadenar múltiples llamadas al modelo donde la salida de una alimenta la siguiente.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Chain 1: Generar un resumen
prompt_resumen = ChatPromptTemplate.from_messages([
    ("system", "Resume el siguiente texto en una sola oración."),
    ("human", "{texto}"),
])

# Chain 2: Traducir al inglés
prompt_traducir = ChatPromptTemplate.from_messages([
    ("system", "Traduce el siguiente texto al inglés."),
    ("human", "{texto}"),
])

# Componer: texto → resumen → traducción
chain_resumen = prompt_resumen | llm | StrOutputParser()
chain_traducir = prompt_traducir | llm | StrOutputParser()

# Ejecutar secuencialmente
texto_largo = """
Colombia es un país ubicado en la esquina noroccidental de América del Sur. 
Es el único país sudamericano con costas tanto en el océano Pacífico como en el Caribe. 
Su capital es Bogotá y tiene una población de aproximadamente 52 millones de habitantes.
Es conocido por su biodiversidad, café, y rica herencia cultural.
"""

resumen = chain_resumen.invoke({"texto": texto_largo})
print(f"Resumen: {resumen}\n")

traduccion = chain_traducir.invoke({"texto": resumen})
print(f"Traducción: {traduccion}")

## 2.5 Batch: procesar múltiples entradas

Podemos enviar varias entradas a la vez.

In [ ]:
chain_simple = prompt | llm | StrOutputParser()

# Procesar varias preguntas
preguntas = [
    {"tema": "gastronomía", "pregunta": "¿Qué es una bandeja paisa?"},
    {"tema": "música", "pregunta": "¿Qué es el vallenato?"},
    {"tema": "geografía", "pregunta": "¿Qué es el Eje Cafetero?"},
]

# batch ejecuta todas en paralelo (si el modelo lo permite)
resultados = chain_simple.batch(preguntas)

for p, r in zip(preguntas, resultados):
    print(f"\n--- {p['pregunta']} ---")
    print(r[:200])

## Ejercicio

1. Crea un `ChatPromptTemplate` que clasifique texto en categorías: positivo, negativo, neutro
2. Usa `with_structured_output` para que devuelva un modelo Pydantic con campos `sentimiento` y `justificacion`
3. Pruébalo con 3 textos diferentes usando `batch`

In [ ]:
# Tu código aquí
